# loading raw files from source azure container to delta tables in bronze 

## call_activity

In [0]:
file_path = "/Volumes/cdl/bronze/landing_files/call_activity_2026_06_15.json"

call_activity_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .json(file_path)
)

display(call_activity_df)

In [0]:
bronze_path = "abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/bronze/call_activity/"

(
    call_activity_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

In [0]:
%sql
CREATE TABLE cdl.bronze.call_activity
USING DELTA
LOCATION 'abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/bronze/call_activity/';

In [0]:
%sql
select * from cdl.bronze.call_activity;

In [0]:
%sql
LIST '/Volumes/cdl/bronze/landing_files/';

In [0]:
%sql
select * from cdl.bronze.call_activity


## Defining a manual schema

In [0]:
%run /Workspace/Users/devesh.ud@gmail.com/pharma_commercial_data_lake/code/config_variables

In [0]:
print(f"{source_base_path}/call_activity_2026_06_15.json")

In [0]:
from pyspark.sql.types import *

call_activity_schema = StructType([
    StructField("call_date", StringType(), True),
    StructField("call_id", StringType(), True),
    StructField("call_status", StringType(), True),
    StructField("call_type", StringType(), True),
    StructField("duration_minutes", LongType(), True)])

df_call = spark.read.format("json")\
    .option("header", True)\
    .schema(call_activity_schema)\
    .load(f"{source_base_path}/call_activity_2026_06_15.json")

display(df_call)

## printing columns / schema

In [0]:
df_call.columns

In [0]:
df_call.printSchema

## Selecting specific columns

In [0]:
df_call.select("call_date","call_id","call_type").display()

In [0]:
from pyspark.sql.functions import *

df_call2 = df_call.select(
    col("call_date").alias("call_dt"),
    col("call_id").alias("event_id"),
    (col("duration_minutes")/60).alias("duration_hours"))

df_call2.display()

##Filtering the data

In [0]:
df_filtered = df_call.filter("call_type = 'Virtual'" + " and call_status = 'Completed'")
df_filtered.display()

In [0]:
from pyspark.sql.functions import *

df_filter = df_call.filter((col("call_type") == "Virtual") | (col("call_type") == "In-person"))
display(df_filter)


In [0]:
df_filter2 = df_call.filter((col("duration_minutes") > 30 ))
display(df_filter2)

In [0]:
df_call.count()
df_filter3 = df_call.filter(~(col("duration_minutes")==45))
df_filter3.count()

In [0]:
df_call.filter(col("call_type").isin("Virtual","In-person")).display()


## Group by and aggregates

In [0]:
# df_call.display()

df_call.groupBy(col("call_type")).count().display()

In [0]:
df_call.groupBy(col("call_status")).agg(count(col("call_id")).alias("total_calls")).display()

In [0]:
# df_call.display()
df_call.groupBy(col("call_status"),col("call_type")).agg(sum(col("duration_minutes")).alias("total_duration")).display()

In [0]:
# df_call.display()

df_call.groupBy(col("call_status"))\
    .agg(count(col("call_id")).alias("total_calls"),\
        sum(col("duration_minutes")).alias("total_duration")).display()

##Drop duplicates

In [0]:
df_call.display()
# df_call.dropDuplicates(["call_date","call_status"]).display()


In [0]:
df_call.groupBy(col("call_status")).agg(count(col("call_id")).alias("total_calls")).filter(col("total_calls") >= 2).display()

In [0]:
df_call.select(col("call_status")).distinct().display()